# FibRec, FibMemo and FibIt (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/FibRec_FibMemo_FibIt_java.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `FibRec`, `FibMemo` and `FibIt`: the algorithms in Java, run on the examples of the chapter, with the «Try it» cell to run them on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/FibRec_FibMemo_FibIt_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/FibRec_FibMemo_FibIt_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `fibRec` (camelCase).

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Fibonacci: FibRec, FibMemo and FibIt (overlapping subproblems)

*Input*: a natural number $n$. *Output*: the Fibonacci number $F(n)$, with $F(0)=0$, $F(1)=1$, $F(n)=F(n-1)+F(n-2)$.

- **FibRec**$(n)$ — direct translation of the definition: the subproblems overlap and the calls are $2F(n+1)-1=\Theta(\varphi^n)$, exponential; the depth is $n$.
- **FibMemo**$(n,memo)$ — recursion with memoization: every value is computed once (the vector `memo` is $-1$ for "undefined"); $2n-1$ calls, memory $\Theta(n)$.
- **FibIt**$(n)$ — loop with two variables: $n-1$ iterations, memory $\Theta(1)$.

On the example $\texttt{FibRec}(5)$ of the chapter we count how many times each subproblem is called (the tree has $15$ nodes, but only $6$ different values are needed). Then the table of the counts (which coincide with the values of the chapter: $21\,891$, $242\,785$, $2\,692\,537$ calls for $n=20,25,30$) and the measured times.

In [2]:
%%writefile Fibonacci.java
import java.util.*;

public class Fibonacci {
    static long fibCalls = 0;   // calls executed

    // fibRec(n)
    static long fibRec(int n) {
        fibCalls++;
        if (n <= 1) {
            return n;
        }
        return fibRec(n - 1) + fibRec(n - 2);
    }

    // fibMemo(n, memo): memo[k] = -1 means "undefined"
    static long fibMemo(int n, long[] memo) {
        fibCalls++;
        if (n <= 1) {
            return n;
        }
        if (memo[n] == -1) {
            memo[n] = fibMemo(n - 1, memo) + fibMemo(n - 2, memo);
        }
        return memo[n];
    }

    // fibIt(n)
    static long fibIt(int n) {
        if (n <= 1) {
            return n;
        }
        long prev = 0;                  // F(i-2)
        long curr = 1;                  // F(i-1)
        int i = 2;
        while (i <= n) {
            long next = prev + curr;
            prev = curr;
            curr = next;
            i = i + 1;
        }
        return curr;
    }

    // Like fibRec, but counts the calls for each value of the parameter
    static long fibRecHist(int n, int[] counts) {
        counts[n]++;
        if (n <= 1) {
            return n;
        }
        return fibRecHist(n - 1, counts) + fibRecHist(n - 2, counts);
    }

    static double nowMs() {
        return System.nanoTime() / 1e6;
    }

    public static void main(String[] args) {
        int[] counts = new int[6];
        long f5 = fibRecHist(5, counts);
        System.out.println("FibRec(5) = " + f5);
        StringBuilder line = new StringBuilder();
        int total = 0;
        for (int k = 5; k >= 0; k--) {
            if (k < 5) line.append("  ");
            line.append(k).append(":").append(counts[k]);
            total += counts[k];
        }
        System.out.println("calls for each value of the parameter: " + line + "   total " + total);
        System.out.println();
        System.out.printf("%3s %8s %9s %10s %8s %6s  (calls, calls, calls, iterations)%n", "n", "F(n)", "FibRec", "2F(n+1)-1", "FibMemo", "FibIt");
        int[] ns = {10, 15, 20, 25, 30};
        for (int n : ns) {
            long f = fibIt(n);
            fibCalls = 0;
            long r1 = fibRec(n);
            long c1 = fibCalls;
            fibCalls = 0;
            long[] memo = new long[n + 1];
            Arrays.fill(memo, -1);
            long r2 = fibMemo(n, memo);
            long c2 = fibCalls;
            boolean ok = (r1 == r2 && r2 == f && c1 == 2 * fibIt(n + 1) - 1 && c2 == 2 * n - 1);
            System.out.printf("%3d %8d %9d %10d %8d %6d  %s%n", n, f, c1, 2 * fibIt(n + 1) - 1, c2, n - 1, ok ? "OK" : "ERROR");
        }
        System.out.println();
        System.out.println("Measured times (they depend on the machine):");
        for (int n : new int[]{20, 25, 30}) {
            double t0 = nowMs(); fibRec(n); double t1 = nowMs() - t0;
            t0 = nowMs();
            long sink = 0;
            for (int rep = 0; rep < 1000; rep++) {
                sink += fibIt(n);
            }
            double t2 = (nowMs() - t0) / 1000;
            System.out.printf("n=%-3d FibRec %10.4f ms   FibIt %10.6f ms%n", n, t1, t2);
        }
    }
}

Writing Fibonacci.java


In [3]:
!javac Fibonacci.java && java Fibonacci

FibRec(5) = 5
calls for each value of the parameter: 5:1  4:1  3:2  2:3  1:5  0:3   total 15

  n     F(n)    FibRec  2F(n+1)-1  FibMemo  FibIt  (calls, calls, calls, iterations)
 10       55       177        177       19      9  OK
 15      610      1973       1973       29     14  OK
 20     6765     21891      21891       39     19  OK
 25    75025    242785     242785       49     24  OK
 30   832040   2692537    2692537       59     29  OK

Measured times (they depend on the machine):
n=20  FibRec     0.0552 ms   FibIt   0.002550 ms
n=25  FibRec     0.7073 ms   FibIt   0.000049 ms
n=30  FibRec     6.7343 ms   FibIt   0.000057 ms


**Try it yourself.** Increase $n$ (with $n=32$ `FibRec` makes $7\,049\,155$ calls, `FibMemo` $63$) and see how the time of `FibRec` grows.

In [4]:
%%writefile ProvaFibonacci.java
import java.util.*;

public class ProvaFibonacci {
    public static void main(String[] args) {
        // Increase n: with n = 32 FibRec makes 7 049 155 calls, FibMemo 63.
        int n = 32;
        Fibonacci.fibCalls = 0;
        long r1 = Fibonacci.fibRec(n);
        System.out.println(r1 + " " + Fibonacci.fibCalls);
        Fibonacci.fibCalls = 0;
        long[] memo = new long[n + 1];
        Arrays.fill(memo, -1);
        long r2 = Fibonacci.fibMemo(n, memo);
        System.out.println(r2 + " " + Fibonacci.fibCalls);
    }
}

Writing ProvaFibonacci.java


In [5]:
!javac ProvaFibonacci.java && java ProvaFibonacci

2178309 7049155
2178309 63


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).